In [6]:
# =========================
# Step 10.1 · Params: 评测工具与 I/O 准备
# =========================
from pathlib import Path

# 统一的中间文件目录（只读写 ./tmp）
TMP_DIR = Path("./tmp").resolve()

# 评测使用的截断 K（指标 @K）
K_EVAL = 20

# 数值容差（用于稳定性检查）
ATOL = 1e-8

# 需要评测的方法前缀（之后会逐个评测；本格只写工具）
METHOD_PREFIXES = {
    "Fused3-RA":   "S_fused3_symrow",
    "Tag-SGNS":    "S_tag_symrow",
    "Text-SGNS":   "S_text_symrow",
    "Behavior":    "S_beh_symrow",
    # 经典非嵌入基线会在 10.3 里另行构造（Text-BM25/Tag-PPMI 的 kNN）
}

# 运行环境信息（可选打印）
print(f"[Env] TMP_DIR = {TMP_DIR}")
print(f"[Eval] K_EVAL = {K_EVAL}")


[Env] TMP_DIR = /home/koyo/workspace/recsys/tmp
[Eval] K_EVAL = 20


In [7]:
# =========================
# Step 10.1 · Execute: 评测与I/O工具
# =========================
import json
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, Tuple, List
from collections import defaultdict

def _require(p: Path, desc: str):
    if not p.exists():
        raise FileNotFoundError(f"[FATAL] 缺少 {desc}: {p.as_posix()}")

def load_manifest(prefix: str) -> Tuple[int, List[Path]]:
    """读取某个方法前缀的 manifest，返回 (nodes, parts)。"""
    mans = sorted(TMP_DIR.glob(f"{prefix}_k*_manifest.json"))
    if not mans:
        raise FileNotFoundError(f"[FATAL] 未找到 {prefix} 的 manifest")
    mp = mans[0]
    man = json.loads(mp.read_text())
    parts = [TMP_DIR / p for p in man["parts"]]
    nodes = int(man.get("nodes", 0))
    print(f"[MAN] {mp.name}  nodes={nodes:,}  parts={len(parts)}  nnz={int(man.get('nnz',0)):,}")
    return nodes, parts

def build_topk_for_method(prefix: str, k_eval: int) -> Tuple[np.ndarray, np.ndarray]:
    """
    为某相似图（行随机、分片 Parquet）构建每行按权重降序的 Top-k 列表。
    返回：
        nbr_idx: (N, k_eval) 邻居 doc_idx，空位=-1
        nbr_w  : (N, k_eval) 对应权重，空位=0
    说明：
      - 我们的对齐图（*_symrow）每行已有 ≤50 条边，但分片顺序未必排序；
        这里对每行做一次小规模 top-k 选择，保证评测次序正确。
      - 实现采用“逐行合并老的 top-k 和新候选（≤50）再选 k_eval”的策略，内存友好。
    """
    N, parts = load_manifest(prefix)
    # 预分配：每行 k_eval 个槽位
    nbr_idx = np.full((N, k_eval), -1, dtype=np.int64)
    nbr_w   = np.zeros((N, k_eval), dtype=np.float32)

    # 为每行维护“已收集数量”计数（最多 50 以内），用于合并比较
    # 实际合并时直接拼接已有 k_eval + 新候选，做一次 argpartition 取前 k_eval
    for i, fp in enumerate(parts, 1):
        df = pd.read_parquet(fp, engine="fastparquet")  # 必有列: row, col, val
        # 用 groupby 将同一行的候选一起处理
        for row, g in df.groupby("row", sort=False):
            row = int(row)
            cand_c = g["col"].to_numpy(np.int64, copy=False)
            cand_w = g["val"].to_numpy(np.float32, copy=False)

            # 取已有 + 新候选 合并再选 top-k
            old_c = nbr_idx[row]
            old_w = nbr_w[row]
            # 拼接（过滤空位）
            mask_old = old_c >= 0
            if mask_old.any():
                cc = np.concatenate([old_c[mask_old], cand_c])
                ww = np.concatenate([old_w[mask_old], cand_w])
            else:
                cc, ww = cand_c, cand_w

            # 若出现重复邻居，做聚合（取最大权重）
            if cc.size > 1:
                ords = np.argsort(cc, kind="mergesort")
                cc = cc[ords]; ww = ww[ords]
                # 压缩重复 key，取 max
                uniq, start = np.unique(cc, return_index=True)
                maxw = np.maximum.reduceat(ww, start)
                cc, ww = uniq, maxw

            # 选前 k_eval
            if cc.size > k_eval:
                part = np.argpartition(ww, -k_eval)[-k_eval:]
                ord2 = np.argsort(-ww[part])
                sel = part[ord2]
                topc = cc[sel]; topw = ww[sel]
            else:
                # 不足 k_eval 就保留全部，并按权重降序排一下（便于指标）
                ord2 = np.argsort(-ww)
                topc = cc[ord2]; topw = ww[ord2]

            # 回写到固定长度槽位
            nk = min(k_eval, topc.size)
            nbr_idx[row, :nk] = topc[:nk]
            nbr_w[row,  :nk]  = topw[:nk]
            if nk < k_eval:
                nbr_idx[row, nk:] = -1
                nbr_w[row,  nk:]  = 0.0

        if (i % 4) == 0 or i == len(parts):
            print(f"[LOAD] {i}/{len(parts)} parts processed for {prefix}")

        del df

    # 稳健性检查（可选）
    rows_no_neighbor = int(np.sum(nbr_idx[:, 0] == -1))
    print(f"[TOPK] {prefix}: rows with no neighbors in top-{k_eval} = {rows_no_neighbor}")
    return nbr_idx, nbr_w

# ---------- 评价指标（@K） ----------
def dcg_at_k(rels: np.ndarray) -> float:
    """rels: 长度<=K 的增益数组（非负），已按排名次序对齐。"""
    if rels.size == 0:
        return 0.0
    ranks = np.arange(1, rels.size + 1)
    return float(np.sum(rels / np.log2(ranks + 1)))

def ndcg_at_k(gains_sorted: np.ndarray, gains_ideal_sorted: np.ndarray) -> float:
    """gains_sorted 为系统返回顺序的增益；gains_ideal_sorted 为理想排序的增益（均取@K）。"""
    dcg = dcg_at_k(gains_sorted)
    idcg = dcg_at_k(gains_ideal_sorted) if gains_ideal_sorted.size else 0.0
    return float(dcg / (idcg + ATOL))

def average_precision_at_k(rels_binary: np.ndarray) -> float:
    """AP@K（binary 相关）。rels_binary: 0/1 数组，对应排名。"""
    if rels_binary.size == 0:
        return 0.0
    hits = 0
    s = 0.0
    for i, r in enumerate(rels_binary, 1):
        if r > 0:
            hits += 1
            s += hits / i
    return float(s / max(1, int(rels_binary.sum())))

def mrr_at_k(rels_binary: np.ndarray) -> float:
    """MRR@K：第一个相关的位置的倒数（无相关则0）。"""
    pos = np.flatnonzero(rels_binary > 0)
    return float(1.0 / (pos[0] + 1)) if pos.size > 0 else 0.0

def precision_at_k(rels_binary: np.ndarray) -> float:
    K = max(1, rels_binary.size)
    return float(np.sum(rels_binary > 0) / K)

def recall_at_k(rels_binary: np.ndarray, total_rel: int) -> float:
    if total_rel <= 0:
        return 0.0
    return float(np.sum(rels_binary > 0) / total_rel)

print("[Step 10.1] 工具加载完成。后续可用 build_topk_for_method() 得到各方法的 Top-K 邻居用于评测；指标函数已就绪。")


[Step 10.1] 工具加载完成。后续可用 build_topk_for_method() 得到各方法的 Top-K 邻居用于评测；指标函数已就绪。


In [13]:
# =========================
# Step 10.2 · Params: 银标相关性构造与缓存
# =========================
from pathlib import Path

TMP_DIR = Path("./tmp").resolve()

# 输入产物（前面步骤已生成）
DOC_CLEAN_PATH   = TMP_DIR / "doc_clean.parquet"   # 含 doc_idx, Id, tag_list（已标准化为小写）
TAG_VOCAB_PATH   = TMP_DIR / "tag_vocab.parquet"   # 含至少 'tag' 与 'df'（或类似命名）
BEH_BASE_PATH    = TMP_DIR / "beh_base.parquet"    # 含 CreatorUserId, OwnerOrganizationId, 以及 log1p 指标等

# 输出缓存（本步生成）
TAG_DOCS_OUT     = TMP_DIR / "relevance_tag_docs.parquet"   # 每个 doc 的“保留标签列表”（仅 kept tags）
TAG_IDF_OUT      = TMP_DIR / "relevance_tag_idf.parquet"    # tag → df, idf
TAG_MANIFEST     = TMP_DIR / "relevance_tag_manifest.json"  # 统计信息

ORG_REL_OUT      = TMP_DIR / "relevance_org.parquet"        # doc_idx → org_id_int（缺失=-1）
CREATOR_REL_OUT  = TMP_DIR / "relevance_creator.parquet"    # doc_idx → creator_id_int（缺失=-1)

print(f"[Env] TMP_DIR = {TMP_DIR}")
print("[Plan] 将生成三套银标：Tag(graded) / Org(binary) / Creator(binary)")


[Env] TMP_DIR = /home/koyo/workspace/recsys/tmp
[Plan] 将生成三套银标：Tag(graded) / Org(binary) / Creator(binary)


In [15]:
# =========================
# Step 10.2 · Execute (Robust): 构造三套银标相关性并缓存
# =========================
import json
import numpy as np
import pandas as pd
from pathlib import Path

def _require(p: Path, desc: str):
    if not p.exists():
        raise FileNotFoundError(f"[FATAL] 缺少 {desc}: {p.as_posix()}")

_require(DOC_CLEAN_PATH, "doc_clean.parquet")
_require(TAG_VOCAB_PATH, "tag_vocab.parquet")
_require(BEH_BASE_PATH, "beh_base.parquet")

# 可选：优先使用 PPMI 的 D–T（更鲁棒），若不存在则退回到 bin
DT_PPMI_PATH = TMP_DIR / "DT_ppmi.parquet"
DT_BIN_PATH  = TMP_DIR / "DT_bin.parquet"
_use_dt_path = None
if DT_PPMI_PATH.exists():
    _use_dt_path = DT_PPMI_PATH
elif DT_BIN_PATH.exists():
    _use_dt_path = DT_BIN_PATH
else:
    raise FileNotFoundError("[FATAL] 未找到 DT_ppmi.parquet 或 DT_bin.parquet，用于在缺失 tag_list 时恢复每文档的标签列表。")

# ---------- (0) 基础信息 ----------
doc_df = pd.read_parquet(DOC_CLEAN_PATH, engine="fastparquet")
N = len(doc_df)
if "doc_idx" in doc_df.columns:
    assert (doc_df["doc_idx"].to_numpy() == np.arange(N)).all(), "[FATAL] doc_clean 未按 0..N-1 对齐"

beh = pd.read_parquet(BEH_BASE_PATH, engine="fastparquet")
assert len(beh) == N, "[FATAL] beh_base 与 doc_clean 行数不一致"
if "doc_idx" in beh.columns:
    assert (beh["doc_idx"].to_numpy() == np.arange(N)).all(), "[FATAL] beh_base 未按 0..N-1 对齐"

# ---------- (A) Tag-Relevance（graded） ----------
# 读取 tag vocab，识别列名并计算 idf
tag_vocab = pd.read_parquet(TAG_VOCAB_PATH, engine="fastparquet")
cols = {c.lower(): c for c in tag_vocab.columns}

tag_col = None
for cand in ["tag", "term", "token", "label", "name"]:
    if cand in cols:
        tag_col = cols[cand]; break
assert tag_col is not None, f"[FATAL] 未在 {TAG_VOCAB_PATH.name} 找到标签列"

df_col = None
for cand in ["df", "doc_freq", "docs", "count"]:
    if cand in cols:
        df_col = cols[cand]; break
assert df_col is not None, f"[FATAL] 未在 {TAG_VOCAB_PATH.name} 找到 df/doc_freq 列"

keep_col = cols.get("keep", None)
if keep_col is not None:
    tag_vocab = tag_vocab[tag_vocab[keep_col].astype(bool)]

# 统一小写并计算 idf
tag_vocab[tag_col] = tag_vocab[tag_col].astype(str).str.strip().str.lower()
df_vals = tag_vocab[df_col].astype(np.int64).to_numpy()
idf_vals = np.log((N + 1) / (df_vals + 1))
tag_idf_df = pd.DataFrame({
    "tag": tag_vocab[tag_col].astype(str).to_numpy(),
    "df":  df_vals,
    "idf": idf_vals.astype(np.float32)
}).drop_duplicates(subset=["tag"])
kept_tags = set(tag_idf_df["tag"].tolist())

# --- 关键分支：是否存在 doc_clean['tag_list'] ---
def _ensure_list(x):
    if isinstance(x, list):
        return [str(t).strip().lower() for t in x if isinstance(t, (str, int, float))]
    if pd.isna(x):
        return []
    s = str(x)
    if s.startswith("[") and s.endswith("]"):
        s = s.strip("[]")
        parts = [p.strip().strip("'").strip('"') for p in s.split(",")]
        return [p.lower() for p in parts if p]
    return [p.strip().lower() for p in s.split(",") if p.strip()]

if "tag_list" in doc_df.columns:
    # 直接用 doc_clean 的 tag_list，过滤到 kept_tags
    tags_series = doc_df["tag_list"].apply(_ensure_list)
    tags_kept = tags_series.apply(lambda lst: [t for t in lst if t in kept_tags])

else:
    # 无 tag_list：从 D–T 边恢复每文档的 kept_tag 列表
    print(f"[TAG] doc_clean 不含 'tag_list'；改用 { _use_dt_path.name } 恢复每文档标签列表 ...")
    dt = pd.read_parquet(_use_dt_path, engine="fastparquet")  # columns: row, col, val
    # 需要 col→tag 的映射：从 tag_vocab 里构建（按出现顺序即词典索引）
    # 假设 tag_vocab 与 DT 的列索引一致，如不一致需在保存 vocab 时一并保存 'col' 索引。
    # 我们在之前的实现里 vocab 顺序即列索引，这里稳妥起见做一列 'col'：
    if "col" in tag_vocab.columns:
        col_to_tag = dict(zip(tag_vocab["col"].astype(int).to_numpy(), tag_vocab[tag_col].to_numpy()))
    else:
        # 退路：按当前排序视为 0..V-1
        tag_vocab = tag_vocab.reset_index(drop=True)
        col_to_tag = dict(zip(np.arange(len(tag_vocab), dtype=np.int64), tag_vocab[tag_col].to_numpy()))
    # 只保留出现在 kept_tags 的列（若 vocab 与 kept 同源，这一步可省；这里双保险）
    # 先反向得到 tag->col，再以 col 过滤
    tag_to_col = {v: k for k, v in col_to_tag.items()}
    kept_cols = np.array([tag_to_col[t] for t in kept_tags if t in tag_to_col], dtype=np.int64)
    kept_cols_mask = pd.Series(False, index=np.unique(list(col_to_tag.keys())))
    kept_cols_mask.loc[kept_cols] = True

    # 过滤到 kept 列后，按 row 聚合出每行的 tag 列表
    # 如果 kept_cols 不完全覆盖 dt['col'] 的全集，使用布尔过滤更稳
    if "col" in dt.columns:
        dt_kept = dt[dt["col"].isin(kept_cols)]
        grp = dt_kept.groupby("row")["col"].apply(lambda xs: [col_to_tag[int(c)] for c in xs.unique()])
    else:
        raise KeyError("[FATAL] D–T 矩阵缺少 'col' 列")

    # 构建完整 N 行的列表（缺失行补空列表）
    tags_list = [grp.get(i, []) for i in range(N)]
    tags_kept = pd.Series(tags_list, index=np.arange(N, dtype=np.int64))

# 输出 tag 文档列表与 idf
tag_docs_df = pd.DataFrame({
    "doc_idx": np.arange(N, dtype=np.int64),
    "tags": tags_kept
})
n_has = int((tag_docs_df["tags"].apply(len) > 0).sum())

tag_docs_df.to_parquet(TAG_DOCS_OUT, engine="fastparquet", index=False)
tag_idf_df.to_parquet(TAG_IDF_OUT, engine="fastparquet", index=False)

man_tag = {
    "nodes": int(N),
    "docs_with_kept_tag": int(n_has),
    "docs_with_kept_tag_ratio": float(n_has / max(1, N)),
    "vocab_size": int(len(tag_idf_df)),
    "note": "Tag-Relevance 使用共享 kept_tags 的 idf 和作为增益；当 doc_clean 无 tag_list 时从 D–T 矩阵恢复每文档标签。"
}
Path(TAG_MANIFEST).write_text(json.dumps(man_tag, ensure_ascii=False, indent=2))
print(f"[TAG] docs_with_kept_tag={n_has:,}/{N:,}  vocab={len(tag_idf_df):,}")
print(f"[SAVE] {TAG_DOCS_OUT.name}, {TAG_IDF_OUT.name}, {Path(TAG_MANIFEST).name}")

# ---------- (B) Org-Relevance（binary） ----------
org_series = beh["OwnerOrganizationId"]
org_mapped = org_series.fillna(-1).astype(np.int64)
org_rel_df = pd.DataFrame({"doc_idx": np.arange(N, dtype=np.int64), "org_id": org_mapped.to_numpy(np.int64)})
org_rel_df.to_parquet(ORG_REL_OUT, engine="fastparquet", index=False)
print(f"[ORG] unique org (含-1) = {org_rel_df['org_id'].nunique()} | 缺失数={(org_rel_df['org_id'] == -1).sum():,}")
print(f"[SAVE] {ORG_REL_OUT.name}")

# ---------- (C) Creator-Relevance（binary） ----------
creator_series = beh["CreatorUserId"]
creator_mapped = creator_series.fillna(-1).astype(np.int64)
creator_rel_df = pd.DataFrame({"doc_idx": np.arange(N, dtype=np.int64), "creator_id": creator_mapped.to_numpy(np.int64)})
creator_rel_df.to_parquet(CREATOR_REL_OUT, engine="fastparquet", index=False)
print(f"[CREATOR] unique creators (含-1) = {creator_rel_df['creator_id'].nunique()} | 缺失数={(creator_rel_df['creator_id'] == -1).sum():,}")
print(f"[SAVE] {CREATOR_REL_OUT.name}")

print("\n[Step 10.2] DONE: 已生成三套银标缓存（Tag/Org/Creator）。")


[TAG] doc_clean 不含 'tag_list'；改用 DT_ppmi.parquet 恢复每文档标签列表 ...
[TAG] docs_with_kept_tag=214,585/521,735  vocab=394
[SAVE] relevance_tag_docs.parquet, relevance_tag_idf.parquet, relevance_tag_manifest.json
[ORG] unique org (含-1) = 391 | 缺失数=519,156
[SAVE] relevance_org.parquet
[CREATOR] unique creators (含-1) = 192013 | 缺失数=0
[SAVE] relevance_creator.parquet

[Step 10.2] DONE: 已生成三套银标缓存（Tag/Org/Creator）。


In [16]:
# =========================
# Step 10.3 · Params: 评测主方法（Fused3-RA / Text-SGNS / Tag-SGNS / Behavior）
# =========================
from pathlib import Path

TMP_DIR = Path("./tmp").resolve()

# 评测截断 @K
K_EVAL = 20

# 本格要评测的方法及其前缀（读取 *_symrow_* 的分片）
METHOD_PREFIXES = {
    "Fused3-RA":   "S_fused3_symrow",
    "Text-SGNS":   "S_text_symrow",
    "Tag-SGNS":    "S_tag_symrow",
    "Behavior":    "S_beh_symrow",
}

# 相关性缓存（10.2 生成）
TAG_DOCS_PATH    = TMP_DIR / "relevance_tag_docs.parquet"    # doc_idx,tags(list[str])
TAG_IDF_PATH     = TMP_DIR / "relevance_tag_idf.parquet"     # tag,df,idf（本格先不启用 idf）
ORG_REL_PATH     = TMP_DIR / "relevance_org.parquet"         # doc_idx, org_id
CREATOR_REL_PATH = TMP_DIR / "relevance_creator.parquet"     # doc_idx, creator_id

# 输出聚合表
OUT_CSV = TMP_DIR / "metrics_main.csv"

print(f"[Env] TMP_DIR={TMP_DIR}")
print(f"[Eval] K_EVAL={K_EVAL}")


[Env] TMP_DIR=/home/koyo/workspace/recsys/tmp
[Eval] K_EVAL=20


In [17]:
# =========================
# Step 10.3 · Execute: 主方法评测
# =========================
import json, numpy as np, pandas as pd
from collections import defaultdict

# ---- 复用 10.1 的工具（确保你已在 10.1 运行过）----
# - build_topk_for_method(prefix, k_eval)
# - 指标函数：ndcg_at_k, average_precision_at_k, mrr_at_k, precision_at_k, recall_at_k

def _require(p: Path, desc: str):
    if not p.exists():
        raise FileNotFoundError(f"[FATAL] 缺少 {desc}: {p.as_posix()}")

_require(TAG_DOCS_PATH,    "relevance_tag_docs.parquet")
_require(ORG_REL_PATH,     "relevance_org.parquet")
_require(CREATOR_REL_PATH, "relevance_creator.parquet")

# ---------- 载入三套银标 ----------
tag_docs = pd.read_parquet(TAG_DOCS_PATH, engine="fastparquet")   # doc_idx, tags(list[str])
N = len(tag_docs)
assert (tag_docs["doc_idx"].to_numpy() == np.arange(N)).all(), "[FATAL] tag_docs 未按 0..N-1 对齐"

org_df = pd.read_parquet(ORG_REL_PATH, engine="fastparquet")      # doc_idx, org_id
creator_df = pd.read_parquet(CREATOR_REL_PATH, engine="fastparquet")  # doc_idx, creator_id
assert len(org_df) == N and len(creator_df) == N, "[FATAL] Org/Creator 长度不一致"

org_ids     = org_df["org_id"].to_numpy(np.int64)
creator_ids = creator_df["creator_id"].to_numpy(np.int64)

# ---------- 为 Tag-Relevance 构建快速判定 ----------
# 把每个 doc 的标签转为 set；同时建立倒排：tag -> set(doc_idx)
doc_tags = [set(lst) if isinstance(lst, list) else set() for lst in tag_docs["tags"].tolist()]
inv_index = defaultdict(set)
for i, tags in enumerate(doc_tags):
    for t in tags:
        inv_index[t].add(i)

# 为 org/creator 预先计算“每个 doc 的总相关数”（Recall 的分母）
# - org：同 org 的文档数 - 1（剔除自身）；org==-1 的样本无可评
# - creator：同 creator 的文档数 - 1（剔除自身）；creator==-1 不会出现（缺失为 0）
def counts_by_id(arr: np.ndarray) -> np.ndarray:
    # 将 id 压缩到 [0..M) 再 bincount
    uniq, inv = np.unique(arr, return_inverse=True)
    cnt = np.bincount(inv)
    return cnt[inv]  # 与 arr 同长

org_cnt_all     = counts_by_id(org_ids)        # 包含自身
creator_cnt_all = counts_by_id(creator_ids)

org_total_rel     = np.where(org_ids >= 0, np.maximum(0, org_cnt_all - 1), 0)
creator_total_rel = np.maximum(0, creator_cnt_all - 1)

# 为 Tag：每个 doc 的“总相关数”（至少共享一个 kept tag）
# 计算方式：对该 doc 的每个 tag，把倒排里的 doc 集合做集合并，最后减去自身
tag_total_rel = np.zeros(N, dtype=np.int32)
for i, tags in enumerate(doc_tags):
    if not tags:
        continue
    # 小集合并（doc 的标签通常很少）
    u = set()
    for t in tags:
        u |= inv_index[t]
    if i in u:
        u.discard(i)
    tag_total_rel[i] = len(u)
del u  # 释放临时变量

# ---------- 评测单个方法 ----------
def evaluate_method(prefix: str, name: str, k_eval: int) -> dict:
    print(f"\n[EVAL] {name} ({prefix}) @K={k_eval}")
    nbr_idx, nbr_w = build_topk_for_method(prefix, k_eval=k_eval)  # (N,K)
    K = nbr_idx.shape[1]

    # 三个任务各自的累计器
    metrics = {
        "nDCG": {"sum": 0.0, "cnt": 0},
        "MAP":  {"sum": 0.0, "cnt": 0},
        "MRR":  {"sum": 0.0, "cnt": 0},
        "P":    {"sum": 0.0, "cnt": 0},
        "R":    {"sum": 0.0, "cnt": 0},
        "COV":  {"sum": 0.0, "cnt": 0},  # 可评覆盖率（有分母的样本占比）
    }
    results = {}

    # ---- (A) Tag-Relevance（二值）----
    ndcg_s = map_s = mrr_s = p_s = r_s = cov_s = 0.0
    cnt = 0
    # 预计算 IDCG 表（binary）：idcg[k] = sum_{i=1..k} 1/log2(i+1)
    idcg_tbl = np.cumsum(1.0 / np.log2(np.arange(2, K+2)))

    for i in range(N):
        # 分母（总相关数）
        tot_rel = tag_total_rel[i]
        if tot_rel <= 0:
            continue  # 不可评
        cov_s += 1.0
        # 邻居二值相关
        neigh = nbr_idx[i]
        rels_bin = np.zeros(K, dtype=np.int8)
        qi_tags = doc_tags[i]
        if qi_tags:
            for j, nb in enumerate(neigh):
                if nb < 0:
                    break
                if (doc_tags[nb] & qi_tags):
                    rels_bin[j] = 1
        # 指标
        # nDCG（二值）：DCG= sum rel/log2(rank+1)，IDCG = min(K,tot_rel) 的前缀和
        dcg = float(np.sum(rels_bin / np.log2(np.arange(2, K+2))))
        idcg = float(idcg_tbl[min(K, tot_rel)-1]) if tot_rel > 0 else 1.0
        ndcg_s += dcg / idcg

        map_s += average_precision_at_k(rels_bin)
        mrr_s += mrr_at_k(rels_bin)
        p_s   += precision_at_k(rels_bin)
        r_s   += recall_at_k(rels_bin, tot_rel)
        cnt   += 1

    if cnt > 0:
        results["Tag-nDCG@{}".format(K)] = ndcg_s / cnt
        results["Tag-MAP@{}".format(K)]  = map_s  / cnt
        results["Tag-MRR@{}".format(K)]  = mrr_s  / cnt
        results["Tag-P@{}".format(K)]    = p_s    / cnt
        results["Tag-R@{}".format(K)]    = r_s    / cnt
        results["Tag-Coverage"]          = cov_s  / N
    else:
        results["Tag-nDCG@{}".format(K)] = results["Tag-MAP@{}".format(K)] = results["Tag-MRR@{}".format(K)] = 0.0
        results["Tag-P@{}".format(K)] = results["Tag-R@{}".format(K)] = results["Tag-Coverage"] = 0.0

    # ---- (B) Org-Relevance（二值）----
    ndcg_s = map_s = mrr_s = p_s = r_s = cov_s = 0.0
    cnt = 0
    for i in range(N):
        if org_ids[i] < 0:
            continue  # 无组织，跳过
        tot_rel = int(org_total_rel[i])
        if tot_rel <= 0:
            continue
        cov_s += 1.0

        neigh = nbr_idx[i]
        rels_bin = (org_ids[neigh.clip(min=0)] == org_ids[i]).astype(np.int8)

        dcg = float(np.sum(rels_bin / np.log2(np.arange(2, K+2))))
        idcg = float(idcg_tbl[min(K, tot_rel)-1])
        ndcg_s += dcg / idcg

        map_s += average_precision_at_k(rels_bin)
        mrr_s += mrr_at_k(rels_bin)
        p_s   += precision_at_k(rels_bin)
        r_s   += recall_at_k(rels_bin, tot_rel)
        cnt   += 1

    if cnt > 0:
        results["Org-nDCG@{}".format(K)] = ndcg_s / cnt
        results["Org-MAP@{}".format(K)]  = map_s  / cnt
        results["Org-MRR@{}".format(K)]  = mrr_s  / cnt
        results["Org-P@{}".format(K)]    = p_s    / cnt
        results["Org-R@{}".format(K)]    = r_s    / cnt
        results["Org-Coverage"]          = cov_s  / N
    else:
        results["Org-nDCG@{}".format(K)] = results["Org-MAP@{}".format(K)] = results["Org-MRR@{}".format(K)] = 0.0
        results["Org-P@{}".format(K)] = results["Org-R@{}".format(K)] = results["Org-Coverage"] = 0.0

    # ---- (C) Creator-Relevance（二值）----
    ndcg_s = map_s = mrr_s = p_s = r_s = cov_s = 0.0
    cnt = 0
    for i in range(N):
        tot_rel = int(creator_total_rel[i])
        if tot_rel <= 0:
            continue
        cov_s += 1.0

        neigh = nbr_idx[i]
        rels_bin = (creator_ids[neigh.clip(min=0)] == creator_ids[i]).astype(np.int8)

        dcg = float(np.sum(rels_bin / np.log2(np.arange(2, K+2))))
        idcg = float(idcg_tbl[min(K, tot_rel)-1])
        ndcg_s += dcg / idcg

        map_s += average_precision_at_k(rels_bin)
        mrr_s += mrr_at_k(rels_bin)
        p_s   += precision_at_k(rels_bin)
        r_s   += recall_at_k(rels_bin, tot_rel)
        cnt   += 1

    if cnt > 0:
        results["Creator-nDCG@{}".format(K)] = ndcg_s / cnt
        results["Creator-MAP@{}".format(K)]  = map_s  / cnt
        results["Creator-MRR@{}".format(K)]  = mrr_s  / cnt
        results["Creator-P@{}".format(K)]    = p_s    / cnt
        results["Creator-R@{}".format(K)]    = r_s    / cnt
        results["Creator-Coverage"]          = cov_s  / N
    else:
        results["Creator-nDCG@{}".format(K)] = results["Creator-MAP@{}".format(K)] = results["Creator-MRR@{}".format(K)] = 0.0
        results["Creator-P@{}".format(K)] = results["Creator-R@{}".format(K)] = results["Creator-Coverage"] = 0.0

    return results

# ---------- 主循环：评每个方法 ----------
rows = []
for name, prefix in METHOD_PREFIXES.items():
    res = evaluate_method(prefix, name, k_eval=K_EVAL)
    row = {"method": name}
    row.update(res)
    rows.append(row)
    # 简短打印
    pretty = ", ".join([f"{k}={v:.4f}" for k, v in res.items() if "Coverage" not in k])
    print(f"[RESULT] {name}: {pretty}")

metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(OUT_CSV, index=False)
print(f"\n[SAVE] {OUT_CSV}")
metrics_df



[EVAL] Fused3-RA (S_fused3_symrow) @K=20
[MAN] S_fused3_symrow_k50_manifest.json  nodes=521,735  parts=14  nnz=26,086,750
[LOAD] 4/14 parts processed for S_fused3_symrow
[LOAD] 8/14 parts processed for S_fused3_symrow
[LOAD] 12/14 parts processed for S_fused3_symrow
[LOAD] 14/14 parts processed for S_fused3_symrow
[TOPK] S_fused3_symrow: rows with no neighbors in top-20 = 0
[RESULT] Fused3-RA: Tag-nDCG@20=0.1097, Tag-MAP@20=0.2358, Tag-MRR@20=0.2667, Tag-P@20=0.0951, Tag-R@20=0.0002, Org-nDCG@20=0.4173, Org-MAP@20=0.5855, Org-MRR@20=0.5998, Org-P@20=0.2290, Org-R@20=0.3035, Creator-nDCG@20=0.8330, Creator-MAP@20=0.8457, Creator-MRR@20=0.8370, Creator-P@20=0.3132, Creator-R@20=0.7890

[EVAL] Text-SGNS (S_text_symrow) @K=20
[MAN] S_text_symrow_k50_manifest.json  nodes=521,735  parts=15  nnz=28,161,106
[LOAD] 4/15 parts processed for S_text_symrow
[LOAD] 8/15 parts processed for S_text_symrow
[LOAD] 12/15 parts processed for S_text_symrow
[LOAD] 15/15 parts processed for S_text_symrow
[T

,method,Tag-nDCG@20,Tag-MAP@20,Tag-MRR@20,Tag-P@20,Tag-R@20,Tag-Coverage,Org-nDCG@20,Org-MAP@20,Org-MRR@20,Org-P@20,Org-R@20,Org-Coverage,Creator-nDCG@20,Creator-MAP@20,Creator-MRR@20,Creator-P@20,Creator-R@20,Creator-Coverage
0,Fused3-RA,0.109729,0.235791,0.266732,0.095071,0.000211,0.411291,0.417277,0.585469,0.599784,0.229043,0.303511,0.004504,0.832993,0.845705,0.836959,0.313248,0.789030,0.7759
1,Text-SGNS,0.029731,0.080014,0.088997,0.029632,0.000038,0.411291,0.000270,0.001056,0.001056,0.000255,0.000036,0.004504,0.000339,0.001103,0.001129,0.000338,0.000029,0.7759
2,Tag-SGNS,0.029952,0.080410,0.089441,0.029879,0.000038,0.411291,0.000439,0.001892,0.001892,0.000362,0.000062,0.004504,0.000361,0.001196,0.001222,0.000353,0.000034,0.7759
3,Behavior,0.135917,0.252093,0.296463,0.121639,0.000270,0.411291,0.476641,0.613048,0.645976,0.287532,0.324898,0.004504,0.865362,0.878008,0.877506,0.342045,0.795286,0.7759


In [18]:
# =========================
# Step 10.4A · Params: Tag-PPMI Cosine & Engagement-Cosine baselines
# =========================
from pathlib import Path
TMP_DIR = Path("./tmp").resolve()

# —— 通用设置 ——
K_TOPK = 50                 # 生成的邻接图每行最多保留 K 条边（保持与前面一致）
PART_EDGES = 2_000_000      # 分片写出时每片的边数目标
K_EVAL = 20                 # 评测截断 @K

# —— 输入（已有产物） ——
DT_PPMI_PATH  = TMP_DIR / "DT_ppmi.parquet"         # D–T (PPMI) 稀疏：row, col, val
TEXT_VOCAB    = TMP_DIR / "tag_vocab.parquet"       # 用于校验/统计（可选）
S_ENG_TOPK_MANIFEST = TMP_DIR / "S_eng_topk_k50_manifest.json"  # 行内 topK（行为特征 FAISS 已算好）

# —— 输出前缀 ——
PPMI_TOPK_PREFIX   = "S_tagppmi_topk"      # 本步先算“topk”，再统一对称化→symrow
PPMI_SYMROW_PREFIX = "S_tagppmi_symrow"
ENG_SYMROW_PREFIX  = "S_engcos_symrow"     # 从 S_eng_topk 派生

print(f"[Env] TMP_DIR={TMP_DIR}")


[Env] TMP_DIR=/home/koyo/workspace/recsys/tmp


In [24]:
# =========================
# Step 10.4 · Params: Tag→graded + Unified 目标 + 可视化
# =========================
from pathlib import Path

TMP_DIR = Path("./tmp").resolve()

# 评测截断
K_EVAL = 20

# 比较的方法（与之前一致）
METHOD_PREFIXES = {
    "Fused3-RA":   "S_fused3_symrow",
    "Text-SGNS":   "S_text_symrow",
    "Tag-SGNS":    "S_tag_symrow",
    "Behavior":    "S_beh_symrow",
}

# 10.2 产物（银标）
TAG_DOCS_PATH    = TMP_DIR / "relevance_tag_docs.parquet"    # doc_idx, tags(list[str])
TAG_IDF_PATH     = TMP_DIR / "relevance_tag_idf.parquet"     # tag, df, idf
ORG_REL_PATH     = TMP_DIR / "relevance_org.parquet"         # doc_idx, org_id
CREATOR_REL_PATH = TMP_DIR / "relevance_creator.parquet"     # doc_idx, creator_id

# Unified 目标的权重（可调，和为1）
W_TAG      = 0.6   # Tag 贡献（graded）
W_ORG      = 0.1   # Org 同组织（binary）
W_CREATOR  = 0.3   # Creator 同人（binary）

# 输出
OUT_CSV_V2 = TMP_DIR / "metrics_main_v2.csv"
OUT_FIG    = TMP_DIR / "fig_ndcg20_main_v2.png"

print(f"[Env] TMP_DIR={TMP_DIR}")
print(f"[Eval] K_EVAL={K_EVAL}, Unified Weights: tag={W_TAG}, org={W_ORG}, creator={W_CREATOR}")


[Env] TMP_DIR=/home/koyo/workspace/recsys/tmp
[Eval] K_EVAL=20, Unified Weights: tag=0.6, org=0.1, creator=0.3


In [ ]:
# =========================
# Step 10.4 · Execute: Tag→graded + Unified + 可视化
# =========================
import numpy as np, pandas as pd, json
from collections import defaultdict
from pathlib import Path
import matplotlib.pyplot as plt

# 复用 Step 10.1 的函数：
# - build_topk_for_method(prefix, k_eval)
# - average_precision_at_k / mrr_at_k / precision_at_k / recall_at_k
# - 我们这里重写一个 nDCG（graded）的实现，IDCG 用“候选并集”近似

def _require(p: Path, desc: str):
    if not p.exists():
        raise FileNotFoundError(f"[FATAL] 缺少 {desc}: {p.as_posix()}")

for p, d in [
    (TAG_DOCS_PATH, "relevance_tag_docs.parquet"),
    (TAG_IDF_PATH,  "relevance_tag_idf.parquet"),
    (ORG_REL_PATH,  "relevance_org.parquet"),
    (CREATOR_REL_PATH, "relevance_creator.parquet"),
]:
    _require(p, d)

# ---- 载入银标与辅助表 ----
tag_docs = pd.read_parquet(TAG_DOCS_PATH, engine="fastparquet")
N = len(tag_docs)
assert (tag_docs["doc_idx"].to_numpy() == np.arange(N)).all(), "[FATAL] tag_docs 未按 0..N-1 对齐"

tag_idf_df = pd.read_parquet(TAG_IDF_PATH, engine="fastparquet")
tag2idf = dict(zip(tag_idf_df["tag"].astype(str), tag_idf_df["idf"].astype(float)))

org_df = pd.read_parquet(ORG_REL_PATH, engine="fastparquet")
creator_df = pd.read_parquet(CREATOR_REL_PATH, engine="fastparquet")
assert len(org_df)==N and len(creator_df)==N
org_ids     = org_df["org_id"].to_numpy(np.int64)
creator_ids = creator_df["creator_id"].to_numpy(np.int64)

# 文档的标签集合 & 其 IDF 总和（用于归一化）
doc_tags = [set(lst) if isinstance(lst, list) else set() for lst in tag_docs["tags"].tolist()]
doc_tag_idf_sum = np.zeros(N, dtype=np.float32)
for i, s in enumerate(doc_tags):
    if s:
        doc_tag_idf_sum[i] = np.sum([tag2idf.get(t, 0.0) for t in s], dtype=np.float32)

# 预计算 org/creator 的相关总数（Recall 分母）
def counts_by_id(arr: np.ndarray) -> np.ndarray:
    uniq, inv = np.unique(arr, return_inverse=True)
    cnt = np.bincount(inv)
    return cnt[inv]

org_cnt_all     = counts_by_id(org_ids)
creator_cnt_all = counts_by_id(creator_ids)
org_total_rel     = np.where(org_ids >= 0, np.maximum(0, org_cnt_all - 1), 0)
creator_total_rel = np.maximum(0, creator_cnt_all - 1)

# ---- 收集各方法的 Top-K 邻居 ----
method_nbr = {}
for name, prefix in METHOD_PREFIXES.items():
    nbr_idx, nbr_w = build_topk_for_method(prefix, k_eval=K_EVAL)
    method_nbr[name] = nbr_idx  # 只用索引即可

# ---- 构建每个 query 的“候选并集” ----
cand_union = []
for i in range(N):
    acc = []
    for name in METHOD_PREFIXES.keys():
        arr = method_nbr[name][i]
        acc.append(arr[arr >= 0])
    if acc:
        cu = np.unique(np.concatenate(acc))
        cu = cu[cu != i]  # 去掉自环
    else:
        cu = np.empty(0, dtype=np.int64)
    cand_union.append(cu)

# ---- 计算单对相似的增益 ----
def tag_gain(i: int, js: np.ndarray) -> np.ndarray:
    """Tag（graded）增益： sum_idf(intersection) / sum_idf(query) ∈ [0,1]，无标签则为0。"""
    if js.size == 0:
        return np.zeros(0, dtype=np.float32)
    denom = float(doc_tag_idf_sum[i])
    if denom <= 0:
        return np.zeros(js.size, dtype=np.float32)
    qi = doc_tags[i]
    gains = np.empty(js.size, dtype=np.float32)
    for k, j in enumerate(js):
        inter = qi & doc_tags[j]
        if not inter:
            gains[k] = 0.0
        else:
            gains[k] = np.sum([tag2idf.get(t, 0.0) for t in inter], dtype=np.float32) / denom
    return gains

def org_bin(i: int, js: np.ndarray) -> np.ndarray:
    if js.size == 0 or org_ids[i] < 0:
        return np.zeros(js.size, dtype=np.float32)
    return (org_ids[js] == org_ids[i]).astype(np.float32)

def creator_bin(i: int, js: np.ndarray) -> np.ndarray:
    if js.size == 0:
        return np.zeros(js.size, dtype=np.float32)
    return (creator_ids[js] == creator_ids[i]).astype(np.float32)

def unified_gain(i: int, js: np.ndarray) -> np.ndarray:
    return W_TAG * tag_gain(i, js) + W_ORG * org_bin(i, js) + W_CREATOR * creator_bin(i, js)

# ---- nDCG(graded) with candidate-union IDCG ----
def ndcg_graded_from_candidates(i: int, ranked_js: np.ndarray, gains_on_rank: np.ndarray,
                                all_js: np.ndarray, gains_all: np.ndarray) -> float:
    """对某 query：已知方法返回顺序 ranked_js 的 graded 增益，以及“候选并集”的所有候选及其增益（用于IDCG）"""
    if ranked_js.size == 0:
        return 0.0
    # DCG
    ranks = np.arange(1, ranked_js.size + 1)
    dcg = float(np.sum(gains_on_rank / np.log2(ranks + 1)))
    # IDCG：候选并集 gains_all 取前K
    if gains_all.size == 0:
        return 0.0
    top = np.sort(gains_all)[-ranked_js.size:][::-1]
    idcg = float(np.sum(top / np.log2(np.arange(1, top.size + 1) + 1)))
    return float(dcg / (idcg + 1e-12))

# ---- 二值指标在候选并集上的实现（用于 MAP/MRR/P/R 的 unified/tag 二值版）----
def binary_from_gain(g: np.ndarray) -> np.ndarray:
    return (g > 0).astype(np.int8)

def eval_suite_for_method(name: str, nbr_idx: np.ndarray) -> dict:
    """返回该方法在四个任务上的聚合指标：Tag(graded)、Org(binary)、Creator(binary)、Unified(graded + binary派生)"""
    K = nbr_idx.shape[1]
    # 预先算折扣
    disc = 1.0 / np.log2(np.arange(2, K + 2))
    idcg_tbl_bin = np.cumsum(disc)  # 二值时的IDCG表

    out = {}

    # ---- A. Tag（graded）----
    ndcg_s = map_s = mrr_s = p_s = r_s = cov = cnt = 0.0
    for i in range(N):
        if doc_tag_idf_sum[i] <= 0:  # 无标签，无从评测
            continue
        ranked = nbr_idx[i]
        ranked = ranked[ranked >= 0]
        if ranked.size == 0:
            continue
        # 本方法排名上的增益
        gains_rank = tag_gain(i, ranked)
        # 候选并集上的增益（用于IDCG）
        allc = cand_union[i]
        gains_all = tag_gain(i, allc)
        # nDCG
        ndcg_s += ndcg_graded_from_candidates(i, ranked, gains_rank, allc, gains_all)
        # 二值派生：>0 为相关
        rel_bin = binary_from_gain(gains_rank)
        # 分母（总相关数）用候选并集的二值个数，避免无限大分母
        tot_rel = int(np.sum(gains_all > 0))
        if tot_rel <= 0:
            continue
        cov += 1.0; cnt += 1.0
        # AP/MRR/P/R（binary）
        map_s += average_precision_at_k(rel_bin)
        mrr_s += mrr_at_k(rel_bin)
        p_s   += precision_at_k(rel_bin)
        r_s   += recall_at_k(rel_bin, tot_rel)

    out[f"TagG-nDCG@{K}"] = ndcg_s / max(1.0, cnt)
    out[f"TagG-MAP@{K}"]  = map_s  / max(1.0, cnt)
    out[f"TagG-MRR@{K}"]  = mrr_s  / max(1.0, cnt)
    out[f"TagG-P@{K}"]    = p_s    / max(1.0, cnt)
    out[f"TagG-R@{K}"]    = r_s    / max(1.0, cnt)
    out["TagG-Coverage"]  = cov / N

    # ---- B. Org（二值，闭式 IDCG，与 10.3 一致）----
    ndcg_s = map_s = mrr_s = p_s = r_s = cov = cnt = 0.0
    for i in range(N):
        if org_ids[i] < 0:  # 无组织
            continue
        tot_rel = int(org_total_rel[i])
        if tot_rel <= 0:
            continue
        ranked = nbr_idx[i]
        ranked = ranked[ranked >= 0]
        if ranked.size == 0:
            continue
        rels = (org_ids[ranked] == org_ids[i]).astype(np.int8)
        # nDCG
        dcg = float(np.sum(rels * disc[:ranked.size]))
        idcg = float(idcg_tbl_bin[min(K, tot_rel) - 1])
        ndcg_s += dcg / idcg
        # 其他
        map_s += average_precision_at_k(rels)
        mrr_s += mrr_at_k(rels)
        p_s   += precision_at_k(rels)
        r_s   += recall_at_k(rels, tot_rel)
        cov += 1.0; cnt += 1.0

    out[f"Org-nDCG@{K}"] = ndcg_s / max(1.0, cnt)
    out[f"Org-MAP@{K}"]  = map_s  / max(1.0, cnt)
    out[f"Org-MRR@{K}"]  = mrr_s  / max(1.0, cnt)
    out[f"Org-P@{K}"]    = p_s    / max(1.0, cnt)
    out[f"Org-R@{K}"]    = r_s    / max(1.0, cnt)
    out["Org-Coverage"]  = cov / N

    # ---- C. Creator（二值，闭式 IDCG，与 10.3 一致）----
    ndcg_s = map_s = mrr_s = p_s = r_s = cov = cnt = 0.0
    for i in range(N):
        tot_rel = int(creator_total_rel[i])
        if tot_rel <= 0:
            continue
        ranked = nbr_idx[i]
        ranked = ranked[ranked >= 0]
        if ranked.size == 0:
            continue
        rels = (creator_ids[ranked] == creator_ids[i]).astype(np.int8)
        dcg = float(np.sum(rels * disc[:ranked.size]))
        idcg = float(idcg_tbl_bin[min(K, tot_rel) - 1])
        ndcg_s += dcg / idcg

        map_s += average_precision_at_k(rels)
        mrr_s += mrr_at_k(rels)
        p_s   += precision_at_k(rels)
        r_s   += recall_at_k(rels, tot_rel)
        cov += 1.0; cnt += 1.0

    out[f"Creator-nDCG@{K}"] = ndcg_s / max(1.0, cnt)
    out[f"Creator-MAP@{K}"]  = map_s  / max(1.0, cnt)
    out[f"Creator-MRR@{K}"]  = mrr_s  / max(1.0, cnt)
    out[f"Creator-P@{K}"]    = p_s    / max(1.0, cnt)
    out[f"Creator-R@{K}"]    = r_s    / max(1.0, cnt)
    out["Creator-Coverage"]  = cov / N

    # ---- D. Unified（graded：W_tag*tag + W_org*bin + W_creator*bin）
    ndcg_s = map_s = mrr_s = p_s = r_s = cov = cnt = 0.0
    for i in range(N):
        ranked = nbr_idx[i]
        ranked = ranked[ranked >= 0]
        if ranked.size == 0:
            continue
        # 增益（本方法排名） & 候选并集上的增益（IDCG）
        gains_rank = unified_gain(i, ranked)
        allc = cand_union[i]
        gains_all = unified_gain(i, allc)
        # 如果候选里所有增益都是 0，则不可评
        if gains_all.size == 0 or np.all(gains_all <= 0):
            continue
        # nDCG（graded）
        ndcg_s += ndcg_graded_from_candidates(i, ranked, gains_rank, allc, gains_all)
        # 二值派生（>0 为相关）
        rel_bin = binary_from_gain(gains_rank)
        tot_rel = int(np.sum(gains_all > 0))
        if tot_rel <= 0:
            continue
        map_s += average_precision_at_k(rel_bin)
        mrr_s += mrr_at_k(rel_bin)
        p_s   += precision_at_k(rel_bin)
        r_s   += recall_at_k(rel_bin, tot_rel)
        cov += 1.0; cnt += 1.0

    out[f"Unified-nDCG@{K}"] = ndcg_s / max(1.0, cnt)
    out[f"Unified-MAP@{K}"]  = map_s  / max(1.0, cnt)
    out[f"Unified-MRR@{K}"]  = mrr_s  / max(1.0, cnt)
    out[f"Unified-P@{K}"]    = p_s    / max(1.0, cnt)
    out[f"Unified-R@{K}"]    = r_s    / max(1.0, cnt)
    out["Unified-Coverage"]  = cov / N

    return out

# ---- 评测并汇总 ----
rows = []
for name, prefix in METHOD_PREFIXES.items():
    res = eval_suite_for_method(name, method_nbr[name])
    row = {"method": name}; row.update(res)
    rows.append(row)
    pretty = ", ".join([f"{k}={v:.4f}" for k, v in res.items() if "Coverage" not in k])
    print(f"[RESULT-v2] {name}: {pretty}")

df_v2 = pd.DataFrame(rows)
df_v2.to_csv(OUT_CSV_V2, index=False)
print(f"\n[SAVE] {OUT_CSV_V2}")
display(df_v2.head())

# ---- 可视化：主图（nDCG@20）----
# 展示四个任务的 nDCG（TagG / Org / Creator / Unified）
tasks = [f"TagG-nDCG@{K_EVAL}", f"Org-nDCG@{K_EVAL}", f"Creator-nDCG@{K_EVAL}", f"Unified-nDCG@{K_EVAL}"]
methods = list(METHOD_PREFIXES.keys())

# 排序方法：按 Unified-nDCG 排序，突出融合优势
order = df_v2.sort_values(by=f"Unified-nDCG@{K_EVAL}", ascending=False)["method"].tolist()
x = np.arange(len(tasks))
width = 0.18

plt.figure(figsize=(10, 5))
for idx, m in enumerate(order):
    vals = [df_v2.loc[df_v2["method"]==m, t].values[0] for t in tasks]
    plt.bar(x + (idx - (len(order)-1)/2)*width, vals, width=width, label=m)

plt.xticks(x, ["Tag (graded)", "Org", "Creator", "Unified"])
plt.ylabel("nDCG@20")
plt.ylim(0, 1.0)
plt.title("Method Comparison (nDCG@20)")
plt.legend()
plt.tight_layout()
plt.savefig(OUT_FIG, dpi=150)
print(f"[FIG] saved -> {OUT_FIG}")


[MAN] S_fused3_symrow_k50_manifest.json  nodes=521,735  parts=14  nnz=26,086,750
[LOAD] 4/14 parts processed for S_fused3_symrow
[LOAD] 8/14 parts processed for S_fused3_symrow
[LOAD] 12/14 parts processed for S_fused3_symrow
[LOAD] 14/14 parts processed for S_fused3_symrow
[TOPK] S_fused3_symrow: rows with no neighbors in top-20 = 0
[MAN] S_text_symrow_k50_manifest.json  nodes=521,735  parts=15  nnz=28,161,106
[LOAD] 4/15 parts processed for S_text_symrow
[LOAD] 8/15 parts processed for S_text_symrow
[LOAD] 12/15 parts processed for S_text_symrow
[LOAD] 15/15 parts processed for S_text_symrow
[TOPK] S_text_symrow: rows with no neighbors in top-20 = 0
[MAN] S_tag_symrow_k50_manifest.json  nodes=521,735  parts=15  nnz=28,159,756
[LOAD] 4/15 parts processed for S_tag_symrow
[LOAD] 8/15 parts processed for S_tag_symrow
[LOAD] 12/15 parts processed for S_tag_symrow
[LOAD] 15/15 parts processed for S_tag_symrow
[TOPK] S_tag_symrow: rows with no neighbors in top-20 = 0
[MAN] S_beh_symrow_k50

In [22]:
# =========================
# Step 10.4A · Execute: Tag-PPMI Cosine & Engagement-Cosine baselines
# =========================
import json
import numpy as np
import pandas as pd
from collections import defaultdict

def _require(p: Path, desc: str):
    if not p.exists():
        raise FileNotFoundError(f"[FATAL] 缺少 {desc}: {p.as_posix()}")

# 复用 10.1/10.3 的工具：build_topk_for_method(), 指标函数（ndcg/map/mrr/p/r）
# 如果当前内核已重启，请先重新运行 10.1 & 10.3 的工具定义单元

# ---------- A) 基线1：Tag-PPMI Cosine ----------
# 思路：用 DT_ppmi 的行向量做余弦相似；为避免全库 O(N^2)，用“候选生成＋累加”近似：
#   · 每个 doc 仅通过它的若干“权重最高的标签”去召回候选；
#   · 同一标签下只取权重最高的前 R 个文档作为候选；
#   · 候选得分 = ∑_t w_i,t * w_j,t；最后除以 ||i|| * ||j|| 做余弦。
DT = pd.read_parquet(DT_PPMI_PATH, engine="fastparquet")  # columns: row, col, val
N = int(DT["row"].max()) + 1

# 每个 doc 的 L2 范数
doc_norm = DT.assign(v2=DT["val"]**2).groupby("row")["v2"].sum().reindex(range(N), fill_value=0).to_numpy()
doc_norm = np.sqrt(np.maximum(doc_norm, 1e-12)).astype(np.float32)

# 每个标签的倒排（按 val 降序），并统计 df
grp_by_col = DT.groupby("col")
post_rows = {}
post_vals = {}
col_df = {}
for col, g in grp_by_col:
    a = g.sort_values("val", ascending=False)
    post_rows[int(col)] = a["row"].to_numpy(np.int32, copy=True)
    post_vals[int(col)] = a["val"].to_numpy(np.float32, copy=True)
    col_df[int(col)] = len(a)

# 每个 doc 取其 top-T 标签作为“召回入口”
T_PER_DOC = 3           # 每个文档使用的最强标签数
R_CAND_EACH = 200       # 每个标签保留的候选上限
DF_CAP = 10000          # df 超过该阈值的热门标签跳过（降低爆炸）

# 为每个 doc 预取 (col, w) 的 top-T
DT_sorted = DT.sort_values(["row", "val"], ascending=[True, False])
topT = DT_sorted.groupby("row").head(T_PER_DOC)
top_map = defaultdict(list)
for r, g in topT.groupby("row"):
    lst = []
    for _, row in g.iterrows():
        c = int(row["col"]); v = float(row["val"])
        if col_df.get(c, 0) <= DF_CAP:
            lst.append((c, v))
    top_map[int(r)] = lst

# 为每个 doc 生成候选并累计分数（dot），再转为余弦并保留 top-K
edge_rows, edge_cols, edge_vals = [], [], []
prog = 0
for i in range(N):
    if (i % 50000) == 0:
        print(f"[PPMI] progressed {i:,}/{N:,}")
    anchors = top_map.get(i, [])
    if not anchors:
        continue
    acc = {}  # neighbor -> dot
    for (c, w_ic) in anchors:
        rows_c = post_rows[c][:R_CAND_EACH]
        vals_c = post_vals[c][:R_CAND_EACH]
        # 累加 dot = sum_t w_i,t * w_j,t
        # 跳过 self
        mask = rows_c != i
        nbrs = rows_c[mask]
        prods = (w_ic * vals_c[mask]).astype(np.float32)
        for nb, sc in zip(nbrs, prods):
            acc[nb] = acc.get(nb, 0.0) + sc
    if not acc:
        continue
    # 余弦：dot / (||i|| * ||j||)
    ni = float(doc_norm[i])
    # 取 top-K
    items = np.fromiter(acc.items(), dtype=object)
    # items: array of tuples; 先拆
    js = np.array([itm[0] for itm in items], dtype=np.int32)
    ds = np.array([itm[1] for itm in items], dtype=np.float32)
    sim = ds / (ni * doc_norm[js])
    if sim.size > K_TOPK:
        part = np.argpartition(sim, -K_TOPK)[-K_TOPK:]
        sel = part[np.argsort(-sim[part])]
        js = js[sel]; sim = sim[sel]
    else:
        ords = np.argsort(-sim)
        js = js[ords]; sim = sim[ords]
    edge_rows.append(np.full(js.size, i, dtype=np.int32))
    edge_cols.append(js.astype(np.int32))
    edge_vals.append(sim.astype(np.float32))

if edge_rows:
    rows = np.concatenate(edge_rows)
    cols = np.concatenate(edge_cols)
    vals = np.concatenate(edge_vals)
else:
    rows = np.zeros(0, dtype=np.int32)
    cols = np.zeros(0, dtype=np.int32)
    vals = np.zeros(0, dtype=np.float32)

print(f"[PPMI] built topK edges = {len(vals):,}")

# 写分片 + manifest
def save_topk(prefix: str, rows: np.ndarray, cols: np.ndarray, vals: np.ndarray, k: int):
    total = len(vals)
    parts = []
    if total == 0:
        raise RuntimeError("[FATAL] no edges")
    start = 0; pid = 0
    while start < total:
        end = min(total, start + PART_EDGES)
        df = pd.DataFrame({"row": rows[start:end], "col": cols[start:end], "val": vals[start:end]})
        fn = f"{prefix}_k{k:02d}_part{pid:04d}.parquet"
        df.to_parquet(TMP_DIR / fn, engine="fastparquet", index=False)
        parts.append(fn)
        print(f"[SAVE] {fn} edges={end-start:,}")
        start = end; pid += 1
    man = {"nodes": int(N), "k": int(k), "parts": parts, "nnz": int(total)}
    (TMP_DIR / f"{prefix}_k{k:02d}_manifest.json").write_text(json.dumps(man, ensure_ascii=False, indent=2))
    print(f"[MANIFEST] {prefix}_k{k:02d}_manifest.json  nnz={total:,}")

save_topk(PPMI_TOPK_PREFIX, rows, cols, vals, K_TOPK)

# ---------- B) 通用：从 topk 对称化 + 行归一，写出 *_symrow_* ----------
def symrow_from_topk(prefix_in: str, prefix_out: str, k: int):
    man = json.loads((TMP_DIR / f"{prefix_in}_k{k:02d}_manifest.json").read_text())
    parts = [TMP_DIR / p for p in man["parts"]]
    # 累计到字典：row -> {col: w_sum}
    adj = defaultdict(dict)
    for i, fp in enumerate(parts, 1):
        df = pd.read_parquet(fp, engine="fastparquet")
        for r, c, v in zip(df["row"].to_numpy(), df["col"].to_numpy(), df["val"].to_numpy()):
            d = adj[r]; d[c] = d.get(c, 0.0) + float(v)   # S
            dT = adj[c]; dT[r] = dT.get(r, 0.0) + float(v) # S^T
        if (i % 4) == 0 or i == len(parts):
            print(f"[LOAD] {i}/{len(parts)} parts loaded for {prefix_in}")

    # 行内 top-k + 归一
    rows_out, cols_out, vals_out = [], [], []
    for r in range(N):
        d = adj.get(r, {})
        if not d:
            continue
        cols_r = np.fromiter(d.keys(), dtype=np.int32)
        vals_r = np.fromiter(d.values(), dtype=np.float32)
        if cols_r.size > k:
            part = np.argpartition(vals_r, -k)[-k:]
            sel = part[np.argsort(-vals_r[part])]
            cols_r = cols_r[sel]; vals_r = vals_r[sel]
        else:
            ords = np.argsort(-vals_r)
            cols_r = cols_r[ords]; vals_r = vals_r[ords]
        s = float(vals_r.sum())
        if s > 0:
            vals_r = (vals_r / s).astype(np.float32)
        rows_out.append(np.full(cols_r.size, r, dtype=np.int32))
        cols_out.append(cols_r.astype(np.int32))
        vals_out.append(vals_r.astype(np.float32))

    rows_out = np.concatenate(rows_out) if rows_out else np.zeros(0, dtype=np.int32)
    cols_out = np.concatenate(cols_out) if cols_out else np.zeros(0, dtype=np.int32)
    vals_out = np.concatenate(vals_out) if vals_out else np.zeros(0, dtype=np.float32)

    save_topk(prefix_out, rows_out, cols_out, vals_out, k)

# 对称化 Tag-PPMI
symrow_from_topk(PPMI_TOPK_PREFIX, PPMI_SYMROW_PREFIX, K_TOPK)

# ---------- C) 基线2：Engagement-Cosine（从 S_eng_topk → symrow） ----------
_require(TMP_DIR / "S_eng_topk_k50_manifest.json", "S_eng_topk_k50_manifest.json")
symrow_from_topk("S_eng_topk", ENG_SYMROW_PREFIX, K_TOPK)

# ---------- D) 评测二个基线 ----------
METHOD_PREFIXES = {
    "Tag-PPMI-Cos":   PPMI_SYMROW_PREFIX,
    "Eng-Cosine":     ENG_SYMROW_PREFIX,
}

rows = []
for name, prefix in METHOD_PREFIXES.items():
    res = evaluate_method(prefix, name, k_eval=K_EVAL)
    row = {"method": name}; row.update(res)
    rows.append(row)
    pretty = ", ".join([f"{k}={v:.4f}" for k, v in res.items() if "Coverage" not in k])
    print(f"[RESULT] {name}: {pretty}")

df_out = pd.DataFrame(rows)
out_csv = TMP_DIR / "metrics_baselines_A.csv"
df_out.to_csv(out_csv, index=False)
print(f"\n[SAVE] {out_csv}")
df_out


KeyboardInterrupt: 